# Kaggle HF Rerun Notebook

Use this notebook in a fresh Kaggle T4x2 session to:

1. clone or update the repository,
2. restore a previously trained adapter from a Kaggle input dataset,
3. run the adapter-backed HF north-star comparison,
4. package the resulting artifacts for download.

This notebook is intentionally smaller than `ft_pipeline.ipynb`. It is for evaluation reruns, not for full training.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from datetime import UTC, datetime
from pathlib import Path

REPO_URL = "https://github.com/aaliyan1230/tool-calling-reliability-benchmark.git"
BRANCH_NAME = "feat/llm"
REPO_DIR = Path("/kaggle/working/tool-calling-reliability-benchmark")
WORKING_DIR = Path("/kaggle/working")
INPUT_DIR = Path("/kaggle/input")
EXPORT_ROOT = WORKING_DIR / "tcrb_kaggle_exports"
RUN_STAMP = datetime.now(UTC).strftime("%Y%m%d-%H%M%S")

# Leave this as None to auto-discover a mounted Kaggle dataset that contains
# either staged_artifacts or a direct adapter export.
# You can still set it manually, for example: INPUT_DIR / "tcrb-kaggle-exports"
ARTIFACT_INPUT_DIR = None

# If your mounted dataset contains a stage export, this notebook will look for
# one of these common layouts.
RESTORE_CANDIDATES = [
    "outputs/ft-notebook/final",
    "outputs/research/qwen25-3b-sft-toolace",
    "staged_artifacts",
]

LABEL_PREFIX = "northstar-hf-qwen25-3b-ft-rerun"
BASE_PLANNER_CONFIG = "configs/planners/hf_qwen2_5_3b_base.json"
COMPARISON_PLANNER_CONFIG = "configs/planners/hf_qwen2_5_3b_comparison.json"

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "repo_dir": str(REPO_DIR),
    "input_dir": str(INPUT_DIR),
    "export_root": str(EXPORT_ROOT),
    "artifact_input_dir": None if ARTIFACT_INPUT_DIR is None else str(ARTIFACT_INPUT_DIR),
    "label_prefix": LABEL_PREFIX,
}, indent=2))

## 1. Clone Or Update The Repo

Run this first. If you made new local repo changes, push them before starting this Kaggle session so the latest notebook and configs are available here.

In [ ]:
if REPO_DIR.exists():
    print(f"Repository already exists at {REPO_DIR}")
    subprocess.run(["bash", "-lc", f"cd {REPO_DIR} && git fetch origin && git checkout {BRANCH_NAME} && git pull --ff-only origin {BRANCH_NAME}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH_NAME, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["bash", "-lc", f"cd {REPO_DIR} && git status --short --branch && git log -1 --oneline"], check=True)

## 2. Install Dependencies

In [ ]:
subprocess.run(["bash", "-lc", f"cd {REPO_DIR} && uv sync --extra dev --extra research"], check=True)

## 3. Restore Prior Adapter Artifacts

The target path in the repo is `outputs/ft-notebook/final`, because the checked-in comparison config points there.

In [ ]:
def copytree_if_exists(source: Path, destination: Path) -> bool:
    if not source.exists():
        return False
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
    return True

def looks_like_artifact_root(path: Path) -> bool:
    return (
        (path / "staged_artifacts").exists()
        or (path / "outputs" / "ft-notebook" / "final").exists()
        or (path / "outputs" / "research").exists()
    )

def discover_artifact_root() -> Path:
    candidates = []
    for child in sorted(INPUT_DIR.iterdir()):
        if not child.is_dir():
            continue
        if looks_like_artifact_root(child):
            candidates.append(child)
    if not candidates:
        raise FileNotFoundError(
            "No mounted Kaggle input dataset looked like a TCRB artifact export. "
            "Expected a directory under /kaggle/input containing staged_artifacts or outputs/ft-notebook/final."
        )
    print("Artifact candidates:")
    for candidate in candidates:
        print("-", candidate)
    return candidates[0]

artifact_root = Path(ARTIFACT_INPUT_DIR) if ARTIFACT_INPUT_DIR is not None else discover_artifact_root()
print(f"Using artifact root: {artifact_root}")
if not artifact_root.exists():
    raise FileNotFoundError(f"Artifact input path does not exist: {artifact_root}")

restored = False
direct_adapter = artifact_root / "outputs" / "ft-notebook" / "final"
staged_roots = sorted((artifact_root / "staged_artifacts").glob("*")) if (artifact_root / "staged_artifacts").exists() else []

if copytree_if_exists(direct_adapter, REPO_DIR / "outputs" / "ft-notebook" / "final"):
    print(f"Restored direct adapter from {direct_adapter}")
    restored = True
elif staged_roots:
    latest_stage = staged_roots[-1]
    source_adapter = latest_stage / "outputs" / "ft-notebook" / "final"
    source_research_adapter = latest_stage / "outputs" / "research" / "qwen25-3b-sft-toolace"
    if copytree_if_exists(source_adapter, REPO_DIR / "outputs" / "ft-notebook" / "final"):
        print(f"Restored adapter from staged export {source_adapter}")
        restored = True
    elif copytree_if_exists(source_research_adapter, REPO_DIR / "outputs" / "ft-notebook" / "final"):
        print(f"Restored adapter from research export {source_research_adapter}")
        restored = True
    source_runs = latest_stage / "runs"
    if source_runs.exists():
        copytree_if_exists(source_runs, REPO_DIR / "runs")
        print(f"Restored prior runs from {source_runs}")

if not restored:
    raise FileNotFoundError(
        "Could not find a previous adapter. Expected either "
        "<artifact_input>/outputs/ft-notebook/final or a staged_artifacts export."
    )

subprocess.run(["bash", "-lc", f"cd {REPO_DIR} && find outputs/ft-notebook/final -maxdepth 2 -type f | sort"], check=True)

## 4. Sanity-Check The Adapter-Backed Comparison Config

In [ ]:
comparison_config = json.loads((REPO_DIR / COMPARISON_PLANNER_CONFIG).read_text(encoding="utf-8"))
print(json.dumps(comparison_config, indent=2))

adapter_path = REPO_DIR / comparison_config["adapter_path"]
if not adapter_path.exists():
    raise FileNotFoundError(f"Adapter path missing: {adapter_path}")
print(f"Adapter path exists: {adapter_path}")

## 5. Run The Adapter-Backed HF Comparison

In [ ]:
rerun_command = " ".join([
    "set -euo pipefail",
    f"cd {REPO_DIR}",
    "source .venv/bin/activate",
    "export PYTHONPATH=$PWD/src${PYTHONPATH:+:$PYTHONPATH}",
    "export MPLBACKEND=Agg",
    "export PYTHONUNBUFFERED=1",
    "uv run python scripts/run_northstar_hf.py",
    f"--base-planner-config {BASE_PLANNER_CONFIG}",
    f"--comparison-planner-config {COMPARISON_PLANNER_CONFIG}",
    f"--label-prefix {LABEL_PREFIX}",
    "--run-study-gate",
    "--run-summarize",
])
print(rerun_command)
subprocess.run(["bash", "-lc", rerun_command], check=True)

## 6. Inspect The Key Outputs

In [ ]:
paths = [
    REPO_DIR / f"runs/{LABEL_PREFIX}-delta/delta-ms.md",
    REPO_DIR / f"runs/{LABEL_PREFIX}-study-gate/study_gate.md",
    REPO_DIR / f"runs/{LABEL_PREFIX}-analysis/analysis_summary.md",
]

for path in paths:
    print(f"\n===== {path} =====")
    if path.exists():
        print(path.read_text(encoding="utf-8")[:6000])
    else:
        print("Missing")

## 7. Package Results For Download

In [ ]:
stage_root = EXPORT_ROOT / f"hf_rerun_{RUN_STAMP}"
stage_root.mkdir(parents=True, exist_ok=True)

paths_to_copy = [
    REPO_DIR / f"runs/{LABEL_PREFIX}-base-ms",
    REPO_DIR / f"runs/{LABEL_PREFIX}-comparison-ms",
    REPO_DIR / f"runs/{LABEL_PREFIX}-delta",
    REPO_DIR / f"runs/{LABEL_PREFIX}-matrix",
    REPO_DIR / f"runs/{LABEL_PREFIX}-study-gate",
    REPO_DIR / f"runs/{LABEL_PREFIX}-analysis",
    REPO_DIR / COMPARISON_PLANNER_CONFIG,
]

for source in paths_to_copy:
    if not source.exists():
        print(f"Skip missing path: {source}")
        continue
    target = stage_root / source.relative_to(REPO_DIR)
    if source.is_dir():
        shutil.copytree(source, target, dirs_exist_ok=True)
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
    print(f"Copied: {source} -> {target}")

manifest_path = stage_root / "export_manifest.json"
manifest_path.write_text(json.dumps({
    "run_stamp": RUN_STAMP,
    "label_prefix": LABEL_PREFIX,
    "repo_dir": str(REPO_DIR),
}, indent=2), encoding="utf-8")

zip_path = shutil.make_archive(str(stage_root), "zip", root_dir=str(stage_root))
print("Export directory:", stage_root)
print("ZIP:", zip_path)